# 05 — Modelagem Supervisionada

Este notebook treina um modelo supervisionado para prever `anomalia_label`, que é derivado do `anomalia_score` gerado pelo Isolation Forest.

A regra é manter fora das features os campos de rótulo e auditoria (`anomalia_score`, `anomalia_label`, `split_dataset`, `candidato_anomalia_heuristica`, `candidato_anomalia`).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
 )
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.config import (
    INTERIM_DIR,
    MODELS_DIR,
    PROCESSED_DIR,
    SUPERVISIONADO_CV_FOLDS,
    SUPERVISIONADO_RANDOM_STATE,
    SUPERVISIONADO_TEST_SIZE,
 )

print('Notebook pronto para treinar o modelo supervisionado.')

Notebook pronto para treinar o modelo supervisionado.


In [2]:
# Carrega o dataset já pontuado pelo Isolation Forest
df = pd.read_parquet(INTERIM_DIR / 'dataset_analitico_com_scores.parquet')

print('Linhas:', len(df))
print('Colunas:', len(df.columns))
print()
print('Distribuição do target supervisionado:')
print(df['anomalia_label'].value_counts(dropna=False))
print()
print('Distribuição por split do pipeline não supervisionado:')
print(df['split_dataset'].value_counts(dropna=False))
print()
print('Taxa de anomalias por split:')
print(df.groupby('split_dataset')['anomalia_label'].mean().round(4))

Linhas: 109346
Colunas: 37

Distribuição do target supervisionado:
anomalia_label
False    103861
True       5485
Name: count, dtype: int64

Distribuição por split do pipeline não supervisionado:
split_dataset
train    87476
test     21870
Name: count, dtype: int64

Taxa de anomalias por split:
split_dataset
test     0.0508
train    0.0500
Name: anomalia_label, dtype: float64


In [3]:
target_col = 'anomalia_label'
leak_cols = {
    'anomalia_score',
    'anomalia_label',
    'split_dataset',
    'candidato_anomalia_heuristica',
    'candidato_anomalia',
}

numeric_features = [
    'valor_licitacao',
    'n_participantes',
    'n_itens',
    'valor_total_itens',
    'hhi',
    'top1_share',
    'log_valor_licitacao',
    'ano',
    'dia_semana',
 ]

categorical_features = [
    'modalidade_compra',
    'uf',
    'situacao_licitacao',
    'ano_mes_arquivo',
 ]

feature_columns = numeric_features + categorical_features

missing_features = [col for col in feature_columns if col not in df.columns]
if missing_features:
    raise ValueError(f'Colunas de features ausentes: {missing_features}')

if leak_cols & set(feature_columns):
    raise ValueError('Features vazam colunas de rótulo.')

train_mask = df['split_dataset'] == 'train'
test_mask = df['split_dataset'] == 'test'

X_train = df.loc[train_mask, feature_columns].copy()
y_train = df.loc[train_mask, target_col].astype(int).copy()
X_test = df.loc[test_mask, feature_columns].copy()
y_test = df.loc[test_mask, target_col].astype(int).copy()

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Features:', feature_columns)

Train shape: (87476, 13)
Test shape: (21870, 13)
Features: ['valor_licitacao', 'n_participantes', 'n_itens', 'valor_total_itens', 'hhi', 'top1_share', 'log_valor_licitacao', 'ano', 'dia_semana', 'modalidade_compra', 'uf', 'situacao_licitacao', 'ano_mes_arquivo']


In [4]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
 )

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=SUPERVISIONADO_RANDOM_STATE,
    n_jobs=-1,
 )

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model),
])

cv = StratifiedKFold(
    n_splits=SUPERVISIONADO_CV_FOLDS,
    shuffle=True,
    random_state=SUPERVISIONADO_RANDOM_STATE,
 )

cv_scores = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        'roc_auc': 'roc_auc',
        'average_precision': 'average_precision',
        'f1': 'f1',
        'balanced_accuracy': 'balanced_accuracy',
    },
    return_train_score=False,
    n_jobs=-1,
 )

print('Validação cruzada no treino:')
for metric in ['test_roc_auc', 'test_average_precision', 'test_f1', 'test_balanced_accuracy']:
    values = cv_scores[metric]
    print(f"{metric}: média={values.mean():.4f}  desvio={values.std(ddof=0):.4f}")

Validação cruzada no treino:
test_roc_auc: média=0.9986  desvio=0.0002
test_average_precision: média=0.9767  desvio=0.0030
test_f1: média=0.9080  desvio=0.0088
test_balanced_accuracy: média=0.9686  desvio=0.0040


In [5]:
pipeline.fit(X_train, y_train)

y_proba_test = pipeline.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= 0.5).astype(int)

test_metrics = {
    'roc_auc': roc_auc_score(y_test, y_proba_test),
    'average_precision': average_precision_score(y_test, y_proba_test),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_test),
    'f1': f1_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test, zero_division=0),
    'recall': recall_score(y_test, y_pred_test, zero_division=0),
}

print('Métricas de teste:')
for metric, value in test_metrics.items():
    print(f'{metric}: {value:.4f}')

print()
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred_test))

print()
print('Relatório de classificação:')
print(classification_report(y_test, y_pred_test, zero_division=0))

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / 'modelo_supervisionado_anomalia.joblib'
pred_path = PROCESSED_DIR / 'predicoes_supervisionadas.parquet'
metrics_path = MODELS_DIR / 'metricas_supervisionado_anomalia.json'

dump(pipeline, model_path)

predicoes = df.loc[test_mask, ['numero_licitacao', 'codigo_ug', 'codigo_modalidade_compra', 'modalidade_compra', 'uf', 'anomalia_score', 'anomalia_label', 'split_dataset']].copy()
predicoes['proba_anomalia'] = y_proba_test
predicoes['pred_anomalia'] = y_pred_test
predicoes.to_parquet(pred_path, index=False)

with open(metrics_path, 'w', encoding='utf-8') as arquivo_saida:
    json.dump(test_metrics, arquivo_saida, ensure_ascii=False, indent=2)

print(f'Modelo salvo em: {model_path}')
print(f'Predições salvas em: {pred_path}')
print(f'Métricas salvas em: {metrics_path}')

Métricas de teste:
roc_auc: 0.9991
average_precision: 0.9835
balanced_accuracy: 0.9766
f1: 0.9233
precision: 0.8898
recall: 0.9595

Matriz de confusão:
[[20627   132]
 [   45  1066]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     20759
           1       0.89      0.96      0.92      1111

    accuracy                           0.99     21870
   macro avg       0.94      0.98      0.96     21870
weighted avg       0.99      0.99      0.99     21870

Modelo salvo em: C:\Users\Gabriela\Pmonkey\IA\predicao\models\modelo_supervisionado_anomalia.joblib
Predições salvas em: C:\Users\Gabriela\Pmonkey\IA\predicao\data\processed\predicoes_supervisionadas.parquet
Métricas salvas em: C:\Users\Gabriela\Pmonkey\IA\predicao\models\metricas_supervisionado_anomalia.json


In [6]:
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
feature_importances = pd.DataFrame({
    'feature': feature_names,
    'importance': pipeline.named_steps['model'].feature_importances_,
}).sort_values('importance', ascending=False)

print('Top 15 features:')
print(feature_importances.head(15).to_string(index=False))

display(feature_importances.head(15))

Top 15 features:
                                          feature  importance
                                valor_total_itens    0.172314
                                          n_itens    0.161048
                                  n_participantes    0.135373
                                       top1_share    0.083744
                                  valor_licitacao    0.082436
                              log_valor_licitacao    0.081663
                                              hhi    0.077119
                     situacao_licitacao_Encerrado    0.055354
     modalidade_compra_Pregão - Registro de Preço    0.048094
          modalidade_compra_Dispensa de Licitação    0.041628
                         modalidade_compra_Pregão    0.011147
                     situacao_licitacao_Publicado    0.009549
   modalidade_compra_Inexigibilidade de Licitação    0.005143
situacao_licitacao_Evento de Resultado de Julgame    0.004161
                                       dia_semana    

,feature,importance
3,valor_total_itens,0.172314
2,n_itens,0.161048
1,n_participantes,0.135373
5,top1_share,0.083744
0,valor_licitacao,0.082436
6,log_valor_licitacao,0.081663
4,hhi,0.077119
49,situacao_licitacao_Encerrado,0.055354
17,modalidade_compra_Pregão - Registro de Preço,0.048094
14,modalidade_compra_Dispensa de Licitação,0.041628


## Conclusão

O modelo supervisionado agora usa apenas variáveis observáveis do dataset analítico e o target `anomalia_label` derivado do score do Isolation Forest.

Se quiser, o próximo passo é ajustar o limiar de decisão, comparar com outros modelos e registrar a melhor configuração no repositório.